# 03 · Frenet Multi-seed — fold-split 분산 제거 + simplicity 확정 (EXP #15, LB 0.6888)

직전 = #14 V2 single-seed(LB 0.6878). single-seed OOF에서 V2(38 feat) vs V3(26 feat) 차이(+0.0011)가 진짜 신호인지 fold-split 우연인지 불분명하다.

**가정**: fold-split seed를 여러 개 평균하면 OOF 분산이 줄어 V2 vs V3의 진짜 우열이 드러난다. 그리고 둘이 noise zone 안에서 동률이면 **단순한 쪽(V3, 26 feat)을 채택**하는 것이 generalization에 유리하다(Karpathy simplicity).

**실험**: V0/V2/V3를 seed {42, 7, 123} × 5-fold로 재학습(270 model), mean±std로 집계. V2 vs V3를 Bonferroni-lite(combined_std × √2)로 비교 — strict 우위면 승자, overlap이면 V3.

**결과**: V2 vs V3 Δ=+0.0003(threshold 안) → **V3 채택 → LB 0.6888 (+0.0010)**. single-seed 우위의 대부분이 split 우연이었음을 확인. V3 std(0.0007) < V2(0.0011)로 feature 적은 쪽이 더 안정적.

## 셀 0 — imports + 상수

In [ ]:
import os, time, json
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold

DT, DT_PRED = 0.04, 0.08
N_FOLDS = 5
MINORITY_THRESHOLD = 5.0
HIT_RADIUS = 0.01

# Multi-seed
SEEDS = [42, 7, 123]

# D4 채택 (v3·T 동일)
BEST_SIGMA = 0.0035
BEST_MAX_W = 2.5

# G_S5_T threshold (D1 reference, 본 multi-seed는 reference 측정만 — Phase 1 sanity은 EXP #14에서 이미 통과)
G_S5_T_THR = 0.6430

os.makedirs('results', exist_ok=True)
print(f'SEEDS={SEEDS}, n_seed={len(SEEDS)}')
print(f'D4: σ={BEST_SIGMA*1000}mm, max_w={BEST_MAX_W}')

## 셀 1 — helper (T 노트북 동일)

In [ ]:
def _norm(x):
    return np.linalg.norm(x, axis=-1)


def physics_baseline(traj):
    p0, p_m40 = traj[:, -1, :], traj[:, -2, :]
    v_last = (p0 - p_m40) / DT
    return (p0 + v_last * DT_PRED).astype(np.float32)


def hit_rate(pred, label, r=HIT_RADIUS):
    return float((np.linalg.norm(pred - label, axis=1) < r).mean())


def w_gaussian(e, sigma, max_w, r=HIT_RADIUS):
    return np.clip(1.0 + (max_w - 1.0) * np.exp(-(e - r)**2 / (2 * sigma**2)),
                   0.5, max_w)


def make_splits(minority_int, seed):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    return [(tr, va) for tr, va in skf.split(np.arange(len(minority_int)), minority_int)]


lgbm_params = dict(
    objective='regression_l1', metric='mae',
    learning_rate=0.05, num_leaves=31, min_data_in_leaf=5,
    max_bin=511, n_estimators=500, random_state=42, verbose=-1,
)
print('helper 정의 완료')

## 셀 2 — Drive mount + 데이터 로드 (캐시 우선)

In [ ]:
CACHE_TRAJ_TR = 'traj_train.npy'
CACHE_Y_TR    = 'y_train.npy'
CACHE_TRAJ_TS = 'traj_test.npy'

DATA_DIR = None
if not (os.path.exists(CACHE_TRAJ_TR) and os.path.exists(CACHE_Y_TR) and os.path.exists(CACHE_TRAJ_TS)):
    from google.colab import drive
    drive.mount('/content/drive')
    ZIP_PATH = '/content/drive/MyDrive/open.zip'
    !unzip -q -o "{ZIP_PATH}" -d /content/
    DATA_DIR = '/content/open' if os.path.exists('/content/open/sample_submission.csv') else '/content'
else:
    print('모든 캐시 존재 — Drive mount 생략')

import pandas as pd

if os.path.exists(CACHE_TRAJ_TR) and os.path.exists(CACHE_Y_TR):
    traj_train = np.load(CACHE_TRAJ_TR)
    y_train    = np.load(CACHE_Y_TR)
else:
    labels = pd.read_csv(f'{DATA_DIR}/train_labels.csv')
    train_ids = labels['id'].tolist()
    y_train = labels[['x','y','z']].values.astype(np.float32)
    traj_train = np.empty((len(train_ids), 11, 3), dtype=np.float32)
    for i, tid in enumerate(train_ids):
        df = pd.read_csv(f'{DATA_DIR}/train/{tid}.csv')
        traj_train[i] = df[['x','y','z']].values
    np.save(CACHE_TRAJ_TR, traj_train)
    np.save(CACHE_Y_TR, y_train)

if DATA_DIR is None:
    DATA_DIR = '/content/open' if os.path.exists('/content/open/sample_submission.csv') else '/content'
    if not os.path.exists(f'{DATA_DIR}/sample_submission.csv'):
        from google.colab import drive
        if not os.path.exists('/content/drive/MyDrive'):
            drive.mount('/content/drive')
        !unzip -q -o /content/drive/MyDrive/open.zip -d /content/

sample_sub = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')
test_ids = sample_sub['id'].tolist()

if os.path.exists(CACHE_TRAJ_TS):
    traj_test = np.load(CACHE_TRAJ_TS)
else:
    traj_test = np.empty((len(test_ids), 11, 3), dtype=np.float32)
    for i, tid in enumerate(test_ids):
        df = pd.read_csv(f'{DATA_DIR}/test/{tid}.csv')
        traj_test[i] = df[['x','y','z']].values
    np.save(CACHE_TRAJ_TS, traj_test)

assert traj_train.shape == (10000, 11, 3) and traj_test.shape == (10000, 11, 3)
print(f'train {traj_train.shape}, test {traj_test.shape}')

## 셀 3 — base + Cart residual + Frenet frame + kinematics

In [ ]:
base_train = physics_baseline(traj_train)
base_test  = physics_baseline(traj_test)
residual_cart_train = (y_train - base_train).astype(np.float32)


def build_frenet_batch(v_last, a_sm, fb_thresh=1e-6):
    vn = np.linalg.norm(v_last, axis=1, keepdims=True)
    eL = v_last / (vn + 1e-9)
    a_perp = a_sm - (a_sm * eL).sum(1, keepdims=True) * eL
    apn = np.linalg.norm(a_perp, axis=1, keepdims=True)
    eN_p = a_perp / (apn + 1e-9)
    z_up = np.array([0., 0., 1.]); y_up = np.array([0., 1., 0.])
    n1 = np.linalg.norm(np.cross(eL, z_up), axis=1, keepdims=True)
    eN_fb1 = np.cross(eL, z_up) / (n1 + 1e-9)
    eN_fb2 = np.cross(eL, y_up); n2 = np.linalg.norm(eN_fb2, axis=1, keepdims=True)
    eN_fb  = np.where(n1 < 1e-6, eN_fb2 / (n2 + 1e-9), eN_fb1)
    eN = np.where(apn < fb_thresh, eN_fb, eN_p)
    eB = np.cross(eL, eN)
    eB = eB / (np.linalg.norm(eB, axis=1, keepdims=True) + 1e-9)
    return eL.astype(np.float32), eN.astype(np.float32), eB.astype(np.float32)


def compute_kinematics(traj):
    v = (traj[:, 1:, :] - traj[:, :-1, :]) / DT
    a = (v[:, 1:, :] - v[:, :-1, :]) / DT
    v_last = v[:, -1, :]
    a_last = a[:, -1, :]
    jerk_last = (a[:, -1, :] - a[:, -2, :]) / DT
    w = np.array([1, 2, 3]) / 6
    a_sm = (a[:, -3:, :] * w[None, :, None]).sum(axis=1)
    return v_last.astype(np.float32), a_last.astype(np.float32), jerk_last.astype(np.float32), a_sm.astype(np.float32)


vl_tr, al_tr, jl_tr, asm_tr = compute_kinematics(traj_train)
vl_ts, al_ts, jl_ts, asm_ts = compute_kinematics(traj_test)
eL_tr, eN_tr, eB_tr = build_frenet_batch(vl_tr, asm_tr)
eL_ts, eN_ts, eB_ts = build_frenet_batch(vl_ts, asm_ts)

# inverse sanity
r_L = (residual_cart_train * eL_tr).sum(1)
r_N = (residual_cart_train * eN_tr).sum(1)
r_B = (residual_cart_train * eB_tr).sum(1)
rec = r_L[:, None]*eL_tr + r_N[:, None]*eN_tr + r_B[:, None]*eB_tr
inv_err = np.abs(rec - residual_cart_train).max()
assert inv_err < 1e-5
print(f'frame inverse max err: {inv_err:.2e}')

residual_fren_train = np.stack([r_L, r_N, r_B], axis=1).astype(np.float32)

acc_norm_last_tr = _norm(al_tr)
minority_mask_tr = acc_norm_last_tr >= MINORITY_THRESHOLD
print(f'minority: {minority_mask_tr.sum()}/{len(minority_mask_tr)}')

## 셀 4 — feature builders (V0 v3 31, V2 38, V3 26)

In [ ]:
def build_v3_31(traj):
    p = traj
    v = (p[:, 1:, :] - p[:, :-1, :]) / DT
    a = (v[:, 1:, :] - v[:, :-1, :]) / DT
    v_last, a_last = v[:, -1, :], a[:, -1, :]
    speed_last, acc_norm_last = _norm(v_last), _norm(a_last)
    w = np.array([1, 2, 3]) / 6
    a3 = a[:, -3:, :]
    a_w3 = (a3 * w[None, :, None]).sum(axis=1)
    acc_norm_w3 = _norm(a_w3)
    v_std, a_std = v.std(axis=1), a.std(axis=1)
    seg = _norm(p[:, 1:, :] - p[:, :-1, :])
    path_eff = _norm(p[:, -1, :] - p[:, 0, :]) / (seg.sum(1) + 1e-9)
    p0 = p[:, -1, :]
    distance_r = _norm(p0)
    radial_v = (p0 * v_last).sum(1) / (distance_r + 1e-9)
    v_n = v / (_norm(v)[..., None] + 1e-9)
    cos_consec = (v_n[:, 1:, :] * v_n[:, :-1, :]).sum(-1).clip(-1, 1)
    turn = np.arccos(cos_consec)
    turn_mean, cos_turn_last = turn.mean(1), cos_consec[:, -1]
    va_dot = (v_last * a_last).sum(1) / (speed_last * acc_norm_last + 1e-9)
    jerk_last = (a[:, -1, :] - a[:, -2, :]) / DT
    v_a_dot = (v_last * a_last).sum(1)
    v_cross_a = np.cross(v_last, a_last)
    a_tang_last = v_a_dot / (speed_last + 1e-9)
    a_cent_last = _norm(v_cross_a) / (speed_last + 1e-9)
    speed_seq = _norm(v)
    speed_diff_half = speed_seq[:, 5:].mean(1) - speed_seq[:, :5].mean(1)
    turn_mean_half_diff = turn[:, 4:].mean(1) - turn[:, :4].mean(1)
    return (
        np.column_stack([
            v_last, a_last, speed_last, acc_norm_last,
            a_w3, acc_norm_w3,
            v_std, a_std, path_eff,
            distance_r, radial_v,
            turn_mean, cos_turn_last, va_dot,
            jerk_last,
            a_tang_last, a_cent_last,
            speed_diff_half, turn_mean_half_diff,
        ]).astype(np.float32),
        a_last, jerk_last, a_w3,
    )


X_v3_31_tr, al_b_tr, jl_b_tr, aw3_b_tr = build_v3_31(traj_train)
X_v3_31_ts, al_b_ts, jl_b_ts, aw3_b_ts = build_v3_31(traj_test)
assert X_v3_31_tr.shape == (10000, 31)


def fren_proj(a_last, jerk_last, a_w3, eL, eN, eB):
    """7 Frenet projection: a_N, a_B, j_L, j_N, j_B, aw_L, aw_N"""
    a_N  = (a_last * eN).sum(1)
    a_B  = (a_last * eB).sum(1)
    j_L  = (jerk_last * eL).sum(1)
    j_N  = (jerk_last * eN).sum(1)
    j_B  = (jerk_last * eB).sum(1)
    aw_L = (a_w3 * eL).sum(1)
    aw_N = (a_w3 * eN).sum(1)
    return np.column_stack([a_N, a_B, j_L, j_N, j_B, aw_L, aw_N]).astype(np.float32)


F7_tr = fren_proj(al_b_tr, jl_b_tr, aw3_b_tr, eL_tr, eN_tr, eB_tr)
F7_ts = fren_proj(al_b_ts, jl_b_ts, aw3_b_ts, eL_ts, eN_ts, eB_ts)

# V0 = v3 31 (Cart target)
X_V0_tr = X_v3_31_tr; X_V0_ts = X_v3_31_ts

# V2 = v3 31 + Frenet 7 = 38 (Frenet target)
X_V2_tr = np.concatenate([X_v3_31_tr, F7_tr], axis=1)
X_V2_ts = np.concatenate([X_v3_31_ts, F7_ts], axis=1)

# V3 = 19 scalars (v3 31 minus 12 Cart dirs) + Frenet 7 = 26 (Frenet target)
CART_DIRS_IDX = list(range(0, 6)) + list(range(8, 11)) + list(range(24, 27))  # vx/vy/vz/ax/ay/az_last, ax/ay/az_w3, jerk_x/y/z_last
SCALAR_IDX = [i for i in range(31) if i not in CART_DIRS_IDX]
assert len(SCALAR_IDX) == 19
X_V3_tr = np.concatenate([X_v3_31_tr[:, SCALAR_IDX], F7_tr], axis=1)
X_V3_ts = np.concatenate([X_v3_31_ts[:, SCALAR_IDX], F7_ts], axis=1)

for n, X in [('V0', X_V0_tr), ('V2', X_V2_tr), ('V3', X_V3_tr)]:
    print(f'  {n}: train {X.shape}, test {dict(V0=X_V0_ts, V2=X_V2_ts, V3=X_V3_ts)[n].shape}')

## 셀 5 — Multi-seed loop helper (per variant)

한 변형(V0/V2/V3) × 한 seed에 대해 Phase 1 (no-weight) → Phase 2 (W1) → test 예측. 모든 결과 Cart basis로 통일 반환.

In [ ]:
def inverse_fren_to_cart(resid_fren, eL, eN, eB):
    return (resid_fren[:, 0:1] * eL + resid_fren[:, 1:2] * eN + resid_fren[:, 2:3] * eB)


def run_one_seed(variant_name, X_tr, X_ts, target_type, seed,
                 sigma=BEST_SIGMA, max_w=BEST_MAX_W):
    """Phase 1 (no-weight) → Phase 2 (W1). Return per-seed result dict.
    target_type: 'cart' (V0) or 'fren' (V2, V3)
    """
    folds = make_splits(minority_mask_tr.astype(int), seed=seed)
    if target_type == 'cart':
        target_tr = residual_cart_train
    else:
        target_tr = residual_fren_train

    # ── Phase 1: no-weight ──
    oof_resid = np.full((10000, 3), np.nan, dtype=np.float32)
    for tr_in, val_in in folds:
        for ax in range(3):
            m = lgb.LGBMRegressor(**lgbm_params)
            m.fit(X_tr[tr_in], target_tr[tr_in, ax],
                  eval_set=[(X_tr[val_in], target_tr[val_in, ax])],
                  callbacks=[lgb.early_stopping(50, verbose=False)])
            oof_resid[val_in, ax] = m.predict(X_tr[val_in]).astype(np.float32)
    # convert to Cart for HR / W1 weight
    if target_type == 'cart':
        oof_resid_cart_p1 = oof_resid
    else:
        oof_resid_cart_p1 = inverse_fren_to_cart(oof_resid, eL_tr, eN_tr, eB_tr)
    pred_p1 = base_train + oof_resid_cart_p1
    oof_e = np.linalg.norm(y_train - pred_p1, axis=1).astype(np.float32)
    HR_p1 = hit_rate(pred_p1, y_train)

    # ── Phase 2: W1 weighted ──
    w_tr = w_gaussian(oof_e, sigma, max_w)
    oof_resid_w1 = np.full((10000, 3), np.nan, dtype=np.float32)
    test_resid_w1 = np.zeros((10000, 3), dtype=np.float32)
    for tr_in, val_in in folds:
        for ax in range(3):
            m = lgb.LGBMRegressor(**lgbm_params)
            m.fit(X_tr[tr_in], target_tr[tr_in, ax],
                  sample_weight=w_tr[tr_in],
                  eval_set=[(X_tr[val_in], target_tr[val_in, ax])],
                  callbacks=[lgb.early_stopping(50, verbose=False)])
            oof_resid_w1[val_in, ax] = m.predict(X_tr[val_in]).astype(np.float32)
            test_resid_w1[:, ax] += m.predict(X_ts).astype(np.float32) / N_FOLDS
    if target_type == 'cart':
        oof_resid_cart_p2 = oof_resid_w1
        test_resid_cart  = test_resid_w1
    else:
        oof_resid_cart_p2 = inverse_fren_to_cart(oof_resid_w1, eL_tr, eN_tr, eB_tr)
        test_resid_cart  = inverse_fren_to_cart(test_resid_w1, eL_ts, eN_ts, eB_ts)
    pred_p2 = base_train + oof_resid_cart_p2
    HR_p2 = hit_rate(pred_p2, y_train)
    HR_p2_major = hit_rate(pred_p2[~minority_mask_tr], y_train[~minority_mask_tr])
    HR_p2_minor = hit_rate(pred_p2[ minority_mask_tr], y_train[ minority_mask_tr])

    return dict(
        variant=variant_name, seed=seed, target_type=target_type,
        HR_p1=HR_p1, HR_p2_overall=HR_p2, HR_p2_major=HR_p2_major, HR_p2_minor=HR_p2_minor,
        oof_resid_cart=oof_resid_cart_p2,
        test_resid_cart=test_resid_cart,
    )

print('helper ready')

## 셀 6 — Multi-seed 학습 (V0, V2, V3 × 3 seed = 9 runs)

270 model 학습, 약 14분.

In [ ]:
VARIANT_INPUTS = {
    'V0': (X_V0_tr, X_V0_ts, 'cart'),
    'V2': (X_V2_tr, X_V2_ts, 'fren'),
    'V3': (X_V3_tr, X_V3_ts, 'fren'),
}

all_results = {}     # (variant, seed) -> result dict
test_preds = {}      # variant -> list of test_resid_cart per seed
oof_preds  = {}      # variant -> list of oof_resid_cart per seed

for vname, (X_tr, X_ts, ttype) in VARIANT_INPUTS.items():
    test_preds[vname] = []
    oof_preds[vname]  = []
    for seed in SEEDS:
        t0 = time.time()
        r = run_one_seed(vname, X_tr, X_ts, ttype, seed)
        all_results[(vname, seed)] = r
        test_preds[vname].append(r['test_resid_cart'])
        oof_preds[vname].append(r['oof_resid_cart'])
        # 캐시 저장
        np.save(f'results/t_ms_oof_{vname}_seed{seed}.npy', r['oof_resid_cart'])
        np.save(f'results/t_ms_test_{vname}_seed{seed}.npy', r['test_resid_cart'])
        print(f'  {vname} seed={seed:3d}: P1 HR={r["HR_p1"]:.4f}  '
              f'P2 overall={r["HR_p2_overall"]:.4f}  '
              f'major={r["HR_p2_major"]:.4f}  minor={r["HR_p2_minor"]:.4f}  '
              f'({time.time()-t0:.0f}s)')
print('\nMulti-seed 학습 완료')

## 셀 7 — Multi-seed mean ± std + 3-gate strict 판정

V2/V3 mean overall vs V0 mean으로 3-gate. combined_std = sqrt(V0_std² + Vi_std²). V2 vs V3 비교는 별도 (Bonferroni-lite × √2).

In [ ]:
def aggregate(variant):
    rs = [all_results[(variant, s)] for s in SEEDS]
    o = np.array([r['HR_p2_overall'] for r in rs])
    M = np.array([r['HR_p2_major'] for r in rs])
    m = np.array([r['HR_p2_minor'] for r in rs])
    p1 = np.array([r['HR_p1'] for r in rs])
    return dict(
        overall_mean=float(o.mean()), overall_std=float(o.std(ddof=1)),
        major_mean=float(M.mean()),   major_std=float(M.std(ddof=1)),
        minor_mean=float(m.mean()),   minor_std=float(m.std(ddof=1)),
        p1_mean=float(p1.mean()),     p1_std=float(p1.std(ddof=1)),
        seeds=SEEDS, per_seed_overall=o.tolist(),
    )


agg = {v: aggregate(v) for v in ['V0', 'V2', 'V3']}

print(f'{"변형":<5} {"seeds":<15} {"P1 overall (mean±std)":<24} {"P2 overall (mean±std)":<24} {"P2 major (mean±std)":<24} {"P2 minor (mean±std)":<24}')
print('-' * 130)
for v in ['V0', 'V2', 'V3']:
    a = agg[v]
    print(f'{v:<5} {str(SEEDS):<15} '
          f'{a["p1_mean"]:.4f} ± {a["p1_std"]:.4f}     '
          f'{a["overall_mean"]:.4f} ± {a["overall_std"]:.4f}     '
          f'{a["major_mean"]:.4f} ± {a["major_std"]:.4f}     '
          f'{a["minor_mean"]:.4f} ± {a["minor_std"]:.4f}')

# Per-seed view
print(f'\nPer-seed P2 overall:')
for v in ['V0', 'V2', 'V3']:
    vals = [all_results[(v, s)]['HR_p2_overall'] for s in SEEDS]
    print(f'  {v}: ' + '  '.join(f'seed={s} {h:.4f}' for s, h in zip(SEEDS, vals)))

# 3-gate vs V0
print(f'\n=== 3-gate strict (vs V0 mean ± std, Bonferroni-lite) ===')
v0 = agg['V0']
for v in ['V2', 'V3']:
    a = agg[v]
    d_o = a['overall_mean'] - v0['overall_mean']
    d_M = a['major_mean']   - v0['major_mean']
    d_m = a['minor_mean']   - v0['minor_mean']
    combined_std = (a['overall_std']**2 + v0['overall_std']**2) ** 0.5
    # 2 variants 비교 → √2 bonferroni
    g1_thr = combined_std * (2 ** 0.5)
    g1 = d_o > g1_thr
    g2 = d_M > -0.002
    g3 = d_m > -0.005
    pass3 = g1 and g2 and g3
    print(f'  {v}: Δoverall={d_o:+.4f} (thr {g1_thr:+.4f})  Δmajor={d_M:+.4f}  Δminor={d_m:+.4f}  '
          f'G1={"O" if g1 else "X"} G2={"O" if g2 else "X"} G3={"O" if g3 else "X"}  '
          f'pass={"O" if pass3 else "X"}')

# V2 vs V3 직접 비교
print(f'\n=== V2 vs V3 직접 비교 ===')
d_v2v3 = agg['V2']['overall_mean'] - agg['V3']['overall_mean']
combined_std_v2v3 = (agg['V2']['overall_std']**2 + agg['V3']['overall_std']**2) ** 0.5
thr_v2v3 = combined_std_v2v3 * (2 ** 0.5)
print(f'  V2 overall_mean = {agg["V2"]["overall_mean"]:.4f} ± {agg["V2"]["overall_std"]:.4f}')
print(f'  V3 overall_mean = {agg["V3"]["overall_mean"]:.4f} ± {agg["V3"]["overall_std"]:.4f}')
print(f'  Δ (V2 - V3) = {d_v2v3:+.4f}  threshold (Bonferroni-lite √2) = ±{thr_v2v3:.4f}')
if abs(d_v2v3) > thr_v2v3:
    winner = 'V2' if d_v2v3 > 0 else 'V3'
    print(f'  → {winner} strict > 상대 변형. 채택 {winner}')
    chosen = winner
else:
    print(f'  → Δ가 threshold 안. tiebreak: V3 (simpler, 26 vs 38 feat)')
    chosen = 'V3'
print(f'\n채택 변형: {chosen}')

## 셀 8 — Submission 3종 생성

- `submission_t_v2_ms3.csv` — V2 3-seed test 평균
- `submission_t_v3_ms3.csv` — V3 3-seed test 평균
- `submission_t_v2v3_ms3.csv` — V2 3 + V3 3 = 6-prediction 평균 (bonus, 이종 결합)

In [ ]:
def make_submission(test_resid_avg, name):
    pred_test_abs = base_test + test_resid_avg
    assert pred_test_abs.shape == (10000, 3)
    assert np.isfinite(pred_test_abs).all()
    df = pd.DataFrame({
        'id': test_ids,
        'x': pred_test_abs[:, 0],
        'y': pred_test_abs[:, 1],
        'z': pred_test_abs[:, 2],
    })
    path = f'submission_t_{name}.csv'
    df.to_csv(path, index=False)
    print(f'  {path}: mean range x[{pred_test_abs[:,0].min():.3f},{pred_test_abs[:,0].max():.3f}] '
          f'y[{pred_test_abs[:,1].min():.3f},{pred_test_abs[:,1].max():.3f}] '
          f'z[{pred_test_abs[:,2].min():.3f},{pred_test_abs[:,2].max():.3f}]')
    return path


v2_test_avg = np.mean(test_preds['V2'], axis=0).astype(np.float32)
v3_test_avg = np.mean(test_preds['V3'], axis=0).astype(np.float32)
v2v3_test_avg = (np.sum(test_preds['V2'], axis=0) + np.sum(test_preds['V3'], axis=0)).astype(np.float32) / 6

p_v2 = make_submission(v2_test_avg, 'v2_ms3')
p_v3 = make_submission(v3_test_avg, 'v3_ms3')
p_v2v3 = make_submission(v2v3_test_avg, 'v2v3_ms3')

## 셀 9 — Meta 저장 + Drive 복사 + 로컬 다운로드

In [ ]:
def safe(x):
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.integer): return int(x)
    return x

agg_safe = {k: {kk: safe(vv) for kk, vv in v.items()} for k, v in agg.items()}

meta = {
    'protocol': 'T 라인 multi-seed (V0/V2/V3 × 3 seeds)',
    'seeds': SEEDS, 'n_folds': N_FOLDS,
    'sigma': BEST_SIGMA, 'max_w': BEST_MAX_W,
    'minority_threshold': MINORITY_THRESHOLD,
    'multi_seed_aggregate': agg_safe,
    'chosen_variant': chosen,
    'submission_v2': p_v2,
    'submission_v3': p_v3,
    'submission_v2v3': p_v2v3,
    'inv_err_train': float(inv_err),
}
with open('results/t_ms_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print('results/t_ms_meta.json 저장')

try:
    from google.colab import drive, files
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    DRIVE_BASE = '/content/drive/MyDrive'
    results_drive = os.path.join(DRIVE_BASE, 'results')
    os.makedirs(results_drive, exist_ok=True)
    !cp -r results/* {results_drive}/
    for p in [p_v2, p_v3, p_v2v3]:
        !cp {p} {DRIVE_BASE}/
    files.download(p_v2)
    files.download(p_v3)
    files.download(p_v2v3)
    print('Drive 복사 + 다운로드 완료')
except ImportError:
    print('Colab 환경 아님 — Drive 복사 skip')